# PHASE 1 — MACHINE LEARNING FOUNDATIONS


# Day 06 — ColumnTransformer


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Use `ColumnTransformer` to apply different preprocessing steps to different columns.
- Combine `Pipeline` and `ColumnTransformer` to build a production-grade ML workflow.
- Establish a robust preprocessing template that you will reuse throughout this course.


## 2. Prerequisites
- Day 4 (Imputation, Scaling, Encoding).
- Day 5 (Pipelines).


## 3. Concept
Real-world datasets have a mix of numerical data (Age, Salary) and categorical data (City, Gender).

A regular `Pipeline` applies the exact same steps to every column you give it. If you feed it a dataset with strings and numbers, the `StandardScaler` will crash on the strings, and the `OneHotEncoder` will incorrectly explode the numbers into thousands of binary categories.

`ColumnTransformer` solves this by routing specific columns to specific pipelines.


## 4. Why Does This Matter?
Without `ColumnTransformer`, you have to manually slice your Pandas DataFrame, process the numeric half, process the categorical half, and then try to stitch them back together using `np.hstack()` (like we did on Day 4). This is tedious, error-prone, and almost guarantees bugs when deploying to production.


## 5. Intuition
Think of a recycling plant.
- The truck dumps all the trash (the mixed DataFrame) onto the conveyor belt.
- The **ColumnTransformer** acts as the sorting machine.
- It routes plastic (Numeric columns) down conveyor belt A (the Numeric Pipeline).
- It routes paper (Categorical columns) down conveyor belt B (the Categorical Pipeline).
- Finally, it recombines the processed materials at the end.


## 6. Mathematical Foundation
If $X$ is partitioned into two sets of columns, $X = [X_{num}, X_{cat}]$, then `ColumnTransformer` computes:

$$ X_{processed} = [P_{num}(X_{num}) \oplus P_{cat}(X_{cat})] $$

Where $\oplus$ represents column-wise concatenation. This guarantees that row alignments remain perfectly intact.


## 7. Scikit-learn API
Using `ColumnTransformer` requires passing a list of tuples. Each tuple contains a name, a transformer (or Pipeline), and a list of columns to apply it to:

```python
ColumnTransformer([
    ('name_1', TransformerA(), ['col1', 'col2']),
    ('name_2', TransformerB(), ['col3'])
])
```


## 8. Simple Example
Let's process the exact same dataset from Day 4, but this time using the industry standard approach.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Create messy DataFrame
df = pd.DataFrame({
    'Age': [25, np.nan, 30, 45, 50],
    'Salary': [50000, 60000, 55000, 100000, np.nan],
    'City': ['Paris', 'London', 'London', 'New York', 'Paris'],
    'Target': [0, 1, 0, 1, 1]
})

X = df.drop('Target', axis=1)
y = df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# 2. Define the individual pipelines
numeric_features = ['Age', 'Salary']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_features = ['City']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine them using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 4. Build final model pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# 5. Train and Predict!
full_pipeline.fit(X_train, y_train)
print('Predictions:', full_pipeline.predict(X_test))


## 9. Code Walkthrough
- We defined `numeric_features` and created a pipeline just for them.
- We defined `categorical_features` and created a pipeline just for them (Notice we used a constant imputer in case a city is NaN!).
- We bundled both into a `ColumnTransformer` called `preprocessor`.
- We created a `full_pipeline` that puts the `preprocessor` first, and the `LogisticRegression` second.
- We achieved perfectly safe, robust, deployable ML in ~20 lines of code.


## 10. Experiment
What happens if we pass a DataFrame with columns that we DID NOT specify in the `ColumnTransformer`? Let's find out!


In [ ]:
df_extra = df.copy()
df_extra['Useless_Col'] = ['A', 'B', 'C', 'D', 'E']
X_extra = df_extra.drop('Target', axis=1)

# Let's push it through the preprocessor ONLY (no model)
processed_extra = preprocessor.fit_transform(X_extra)
print('Shape of output:', processed_extra.shape)
print('\nWhy? By default, ColumnTransformer simply DROPS any columns not explicitly mentioned in the transformers list!')


## 11. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
from sklearn.compose import make_column_transformer
col_trans = make_column_transformer(
    (StandardScaler(), ['Age', 'Salary']),
    (OneHotEncoder(), ['City'])
)


> **Question:** Just like `make_pipeline`, `make_column_transformer` is a shortcut. What does it automatically generate for you?

**Think before running the next cell!**


In [ ]:
print('It generates the step names! Instead of writing ("num", StandardScaler(), ["Age"]), you just pass the transformer and the columns.')


## 12. Coding Exercise
Create a dataset `X_ex` with columns `['Height', 'Weight', 'Gender']`. 
Build a `ColumnTransformer` that scales Height and Weight using `MinMaxScaler`, and encodes Gender using `OrdinalEncoder`. 
Apply it using `.fit_transform()`.


In [ ]:
# YOUR CODE HERE
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
X_ex = pd.DataFrame({
    'Height': [170, 180, 160, 190],
    'Weight': [70, 80, 60, 90],
    'Gender': ['M', 'M', 'F', 'M']
})

col_t = ColumnTransformer([
    ('scale', MinMaxScaler(), ['Height', 'Weight']),
    ('encode', OrdinalEncoder(), ['Gender'])
])

print('Result:\n', col_t.fit_transform(X_ex))


## 13. Debugging Challenge
The junior data scientist tried to use `ColumnTransformer` but caused an error. Find the bug!


In [ ]:
# Buggy code
try:
    bug_ct = ColumnTransformer([
        ('numeric', StandardScaler(), ['Age']),
        # BUG IS HERE:
        ('categoric', 'OneHotEncoder', ['City'])
    ])
    # bug_ct.fit_transform(X_train)
except Exception as e:
    print('Error:', e)


> **Hint:** Scikit-learn expects instantiated objects, not strings. (Except for the special string `'passthrough'` or `'drop'`).


## 14. Model Evaluation
A beautifully structured `ColumnTransformer` -> `Pipeline` architecture doesn't just prevent bugs; it makes hyperparameter tuning much easier later on, because we can cleanly search for the best imputation strategies or scaler types simultaneously with model parameters.


## 15. Real-World Example
In real datasets, some features don't need any preprocessing (e.g., a binary flag that is already 0 or 1). 
You can use `remainder='passthrough'` in `ColumnTransformer`. This tells Scikit-learn: *"Process the columns I specified, and for everything else, just pass them through untouched instead of dropping them."*


## 16. Mini Project
Modify the `preprocessor` from Section 8 to use `remainder='passthrough'`. Pass the `df_extra` DataFrame through it and verify the shape increases by 1 column (because 'Useless_Col' is passed through).


In [ ]:
preprocessor_pass = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Do not drop unlisted columns
)

processed_pass = preprocessor_pass.fit_transform(X_extra)
print('Original output shape (dropped):', processed_extra.shape)
print('New output shape (passthrough):', processed_pass.shape)


## 17. Common Mistakes
- **Forgetting the column lists**: Writing `('scaler', StandardScaler())` instead of `('scaler', StandardScaler(), ['Age'])`. `ColumnTransformer` must know *where* to apply the transformer.
- **Passing numpy arrays instead of DataFrames**: If you pass a numpy array to a `ColumnTransformer` that expects string column names (like `'Age'`), it will crash. If passing numpy arrays, you must use integer column indices (e.g., `[0, 1]`).
- **Order of columns**: The output of `ColumnTransformer` places the columns in the order they were processed, which might scramble your original DataFrame's column order. This is completely fine for models, but confusing for humans.


## 18. Interview Questions
- **Beginner**: Why do we use `ColumnTransformer`?
- **Intermediate**: What happens to columns in a DataFrame that are not specified in the `ColumnTransformer`?
- **Advanced**: How would you combine `Pipeline` and `ColumnTransformer` to impute missing categories with a constant string, and then One-Hot Encode them?


## 19. Knowledge Check
- What parameter do you use to keep unlisted columns instead of dropping them? (`remainder='passthrough'`)


## 20. Summary
- **ColumnTransformer** routes specific columns to specific preprocessing steps.
- Output columns are concatenated horizontally (`np.hstack`).
- Unlisted columns are dropped by default.
- The combination of **`ColumnTransformer` + `Pipeline`** is the gold standard architecture for Scikit-learn workflows.


## 21. Homework
Load the `titanic` dataset (or create a dummy version). Create a complete pipeline that scales `Age` and `Fare`, one-hot encodes `Sex` and `Embarked`, passes through `Pclass`, and fits a `LogisticRegression` model.
